In [0]:
%python
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
df = spark.table("covid_19_states").toPandas()
print(df.head())




In [0]:
%python
print(df.describe())

In [0]:
%python
print(df.shape)
# print(df.isnull().sum())
# print(df.dtypes)
print(df[df['Tested'].isnull()])

In [0]:
%python
df["positivity_rate"] = df["Confirmed"] / df["Tested"]
avg_ratio = df["positivity_rate"].mean()
print(avg_ratio)
df.loc[df["Tested"].isnull(), "Tested"] = df["Confirmed"] / avg_ratio
df.drop('positivity_rate', axis=1, inplace=True)

In [0]:
%python
print(df.isnull().sum())
df["Date"] = pd.to_datetime(df["Date"])
df["month"] = df["Date"].dt.month
df["year"] = df["Date"].dt.year
df["day"] = df["Date"].dt.day
print(df.dtypes)

In [0]:
%python
# # Highest confirmed cases for each state and month (excluding India)
# df_filtered = df[df["State"] != "India"].copy()
# df["Date"] = pd.to_datetime(df["Date"])
# df["month"] = df["Date"].dt.month
# idx = df_filtered.groupby(["State", "month"])["Confirmed"].idxmax()
# monthly_max = df_filtered.loc[idx, ["State", "month", "Confirmed", "Date"]]
# monthly_max = monthly_max.sort_values(["State", "month"])
# display(monthly_max.head(30))

In [0]:
%python
spark.createDataFrame(df).write.mode("overwrite").saveAsTable("covid_19_states_cleaned")

In [0]:
CREATE OR REPLACE TABLE covid_19_states_cleaned AS
SELECT 
  Date, State, Confirmed, Recovered, Deceased, Other,
  COALESCE(Tested, Confirmed / 0.059133380805568) AS Tested
FROM covid_19_states;

In [0]:
select * from covid_19_states;


Which state has the highest confirmed cases?

In [0]:
SELECT State, MAX(Confirmed) AS max_confirmed
FROM covid_19_states
GROUP BY State having State!='India'
ORDER BY max_confirmed DESC;

Which state has the highest deaths

In [0]:
SELECT State, MAX(Deceased) AS max_deceased
FROM covid_19_states
GROUP BY State having State!='India'
ORDER BY max_deceased DESC;

Total Cases in India

In [0]:
select State,sum(Confirmed) from covid_19_states GROUP BY State having State='India';

Total Recoveries by each state


In [0]:
select State,sum(Recovered) from covid_19_states GROUP BY State HAVING State!='India';

Months having Biggest Case Spikes.

In [0]:
SELECT month(Date) AS covid_month,MAX(Confirmed) AS max_confirmed 
FROM covid_19_states_cleaned 
GROUP BY covid_month 
ORDER BY covid_month;
-- // Also try to include state to it

States and their Recovery rates

In [0]:
SELECT State, (SUM(Recovered)/SUM(Confirmed))*100 AS recovery_rate FROM covid_19_states_cleaned GROUP BY State HAVING State!='India' ORDER BY recovery_rate DESC 

States and their death rate

In [0]:
SELECT State, (SUM(Deceased)/SUM(Confirmed))*100 AS death_rate FROM covid_19_states_cleaned GROUP BY State HAVING State!='India' ORDER BY death_rate DESC; 

In [0]:
WITH avg_calc AS (
    SELECT AVG(Confirmed * 1.0 / Tested) AS avg_ratio
    FROM covid_19_states_cleaned
    WHERE Tested IS NOT NULL AND Tested != 0
)

UPDATE covid_19_states_cleaned
SET Tested = Confirmed / (SELECT avg_ratio FROM avg_calc)
WHERE Tested IS NULL;

select Tested from covid_19_states_cleaned;

States and their test rate

In [0]:
SELECT State, SUM(Tested) AS tests FROM covid_19_states_cleaned GROUP BY State HAVING State!='India' ORDER BY tests DESC;

Maximum number of confirmed cases by Month

In [0]:
SELECT covid_month, State, Confirmed
FROM (
    SELECT 
        EXTRACT(MONTH FROM Date) AS covid_month,
        State,
        Confirmed,
        RANK() OVER (
            PARTITION BY EXTRACT(MONTH FROM Date) 
            ORDER BY Confirmed DESC
        ) AS rnk
    FROM covid_19_states_cleaned
    WHERE State != 'India'
) ranked
WHERE rnk = 1
ORDER BY covid_month;

In [0]:
SELECT covid_month, State, Recovered
FROM (
    SELECT 
        EXTRACT(MONTH FROM Date) AS covid_month,
        State,
        Recovered,
        RANK() OVER (
            PARTITION BY EXTRACT(MONTH FROM Date) 
            ORDER BY Recovered DESC
        ) AS rnk
    FROM covid_19_states_cleaned
    WHERE State != 'India'
) ranked
WHERE rnk = 1
ORDER BY covid_month;

In [0]:
select * from covid_19_states_cleaned